# Notebook 0 — Data preparation and non-LLM baselines (CPU only)

Produces, in `MyDrive/asag_revision/`:
- `data/mohler_prepared.csv`, `data/scientsbank_prepared.csv` — cleaned data with question-grouped splits
- `results/dataset_stats_*.csv` — dataset statistics and human inter-rater agreement
- `cache/grammar_errors.csv`, `cache/features.csv` — lexical features (resumable)
- `results/mohler_preds.csv`, `results/scientsbank_preds.csv` — per-answer predictions (later notebooks add LLM columns)
- `results/*_nb0_summary.csv`, `results/nb0_weights.csv`, `results/nb0_latency.csv`

**Before running:** upload `asag_utils.py` to `MyDrive/asag_revision/` and `mohler_dataset_edited.csv` to `MyDrive/asag_revision/data/`. Runtime type: CPU is fine.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
ROOT = '/content/drive/MyDrive/asag_revision'
for sub in ['data', 'results', 'cache']:
    os.makedirs(f'{ROOT}/{sub}', exist_ok=True)
sys.path.insert(0, ROOT)

Mounted at /content/drive


In [2]:
!apt-get -qq install -y openjdk-17-jre-headless > /dev/null
!pip -q install sentence-transformers language-tool-python datasets

E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/o/openjdk-17/openjdk-17-jre-headless_17.0.20%2b8-1%7e24.04_amd64.deb  404  Not Found [IP: 91.189.91.82 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 2.4 MB/s eta 0:00:00


In [3]:
import json, time, platform
import numpy as np, pandas as pd, nltk, sklearn
from tqdm.auto import tqdm
for p in ['stopwords', 'wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    nltk.download(p, quiet=True)

from asag_utils import *

versions = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
            'sklearn': sklearn.__version__, 'nltk': nltk.__version__}

## 1. Mohler dataset — 5 question-grouped folds, each with a question-grouped validation set

In [4]:
mo = pd.read_csv(f'{ROOT}/data/mohler_dataset_edited.csv').rename(columns={'desired_answer': 'reference_answer'})
mo['uid'] = [f'm{i}' for i in range(len(mo))]
mo['group'] = mo['question']            # group by question text (81 ids map to 79 distinct questions)
mo['y'] = mo['score_avg'] * 2           # 0-5 mean of two graders -> 0-10 (linear rescaling)
mo['y_int'] = np.clip(np.floor(mo['y'] + 0.5), 0, 10).astype(int)   # integer target for LLM fine-tuning
mo['fold'] = grouped_folds(mo['group'].values)

for k in range(N_FOLDS):
    split = np.where(mo['fold'] == k, 'test', 'train').astype(object)
    rest = (mo['fold'] != k).values
    val = np.zeros(len(mo), bool)
    val[rest] = grouped_holdout(mo.loc[rest, 'group'].values, 0.15, seed=SEED + k)
    split[val] = 'val'
    mo[f'split_f{k}'] = split
    # leakage check: no question may appear in two partitions
    assert mo.groupby('group')[f'split_f{k}'].nunique().max() == 1

mo.to_csv(f'{ROOT}/data/mohler_prepared.csv', index=False)
pd.DataFrame({f'fold {k}': mo[f'split_f{k}'].value_counts() for k in range(N_FOLDS)})

,fold 0,fold 1,fold 2,fold 3,fold 4
train,1575,1535,1550,1538,1557
test,441,466,469,455,442
val,257,272,254,280,274


## 2. SciEntsBank — official SemEval-2013 train / test_ua / test_uq / test_ud, plus a question-grouped validation set

In [5]:
from datasets import load_dataset
raw = load_dataset('nkazi/SciEntsBank')
names = raw['train'].features['label'].names
parts = []
for s in ['train', 'test_ua', 'test_uq', 'test_ud']:
    d = raw[s].to_pandas()
    d['label'] = [names[i] for i in d['label']]
    d['split'] = s
    parts.append(d)
se = pd.concat(parts, ignore_index=True)
se['uid'] = [f's{i}' for i in range(len(se))]
se['group'] = se['question']
tr = (se['split'] == 'train').values
val = np.zeros(len(se), bool)
val[tr] = grouped_holdout(se.loc[tr, 'group'].values, 0.10)
se.loc[val, 'split'] = 'val'
se['label3'] = se['label'].map(to3)
se['label2'] = se['label'].map(to2)

trv = se[se.split.isin(['train', 'val'])]
assert trv.groupby('group')['split'].nunique().max() == 1
se.to_csv(f'{ROOT}/data/scientsbank_prepared.csv', index=False)
pd.crosstab(se['label'], se['split'], margins=True)

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

data/train-00001.parquet: reconstructing file:   0%|          |  0.00B /  233kB            

data/train-00001.parquet: downloading bytes:           |  0.00B            

data/test-ua-00001.parquet: reconstructing file:   0%|          |  0.00B / 52.7kB            

data/test-ua-00001.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/4969 [00:00<?, ? examples/s]

Generating test_ua split:   0%|          | 0/540 [00:00<?, ? examples/s]

Generating test_uq split:   0%|          | 0/733 [00:00<?, ? examples/s]

Generating test_ud split:   0%|          | 0/4562 [00:00<?, ? examples/s]

split,test_ua,test_ud,test_uq,train,val,All
label,,,,,,
contradictory,58,417,64,444,55,1038
correct,233,1917,301,1798,210,4459
irrelevant,133,1222,193,990,125,2663
non_domain,3,20,0,21,2,46
partially_correct_incomplete,113,986,175,1219,105,2598
All,540,4562,733,4472,497,10804


## 3. Dataset statistics and human inter-rater agreement (Mohler)

In [6]:
def n_words(s): return s.astype(str).str.split().str.len()

mo_stats = pd.DataFrame({
    'answers': [len(mo)], 'questions': [mo.group.nunique()],
    'answers_per_q_mean': [mo.groupby('group').size().mean()],
    'student_words_mean': [n_words(mo.student_answer).mean()],
    'reference_words_mean': [n_words(mo.reference_answer).mean()],
    'y_mean': [mo.y.mean()], 'y_std': [mo.y.std()]})
mo_scoredist = mo['y_int'].value_counts().sort_index().rename('count')

se_stats = se.groupby('split').agg(answers=('uid', 'size'), questions=('group', 'nunique'))
se_stats['student_words_mean'] = se.assign(w=n_words(se.student_answer)).groupby('split')['w'].mean()

human = reg_metrics(mo['score_me'] * 2, mo['score_other'] * 2)   # grader 1 vs grader 2, 0-10 scale

mo_stats.to_csv(f'{ROOT}/results/dataset_stats_mohler.csv', index=False)
mo_scoredist.to_csv(f'{ROOT}/results/dataset_scoredist_mohler.csv')
se_stats.to_csv(f'{ROOT}/results/dataset_stats_scientsbank.csv')
pd.DataFrame([human]).to_csv(f'{ROOT}/results/human_interrater_mohler.csv', index=False)
display(mo_stats.round(2), mo_scoredist.to_frame().T, se_stats.round(2))
print('Human inter-rater (Mohler):', {k: round(v, 3) for k, v in human.items()})

,answers,questions,answers_per_q_mean,student_words_mean,reference_words_mean,y_mean,y_std
0,2273,79,28.77,19.06,15.34,8.33,2.21


y_int,0,1,2,3,4,5,6,7,8,9,10
count,23,3,11,43,98,122,155,184,207,308,1119


,answers,questions,student_words_mean
split,,,
test_ua,540,135,12.20
test_ud,4562,45,11.37
test_uq,733,15,12.22
train,4472,121,12.62
val,497,14,12.34


Human inter-rater (Mohler): {'pearson': np.float64(0.597), 'spearman': np.float64(0.601), 'mae': np.float64(1.496), 'rmse': np.float64(2.621), 'qwk': np.float64(0.49)}


## 4. Lexical features (resumable; grammar checking is the slow step, ~15–25 min)

In [7]:
import language_tool_python
from importlib.metadata import version
versions['language_tool_python'] = version('language-tool-python')
tool = language_tool_python.LanguageTool('en-US')

all_ans = pd.concat([mo[['uid', 'student_answer']], se[['uid', 'student_answer']]], ignore_index=True)
gpath = f'{ROOT}/cache/grammar_errors.csv'
done = pd.read_csv(gpath) if os.path.exists(gpath) else pd.DataFrame(columns=['uid', 'n_errors', 'n_typos'])
todo = all_ans[~all_ans.uid.isin(done.uid)]

def is_typo(m):
    return getattr(m, 'ruleIssueType', '') == 'misspelling' or getattr(m, 'category', '') == 'TYPOS'

buf = []
for uid, text in tqdm(zip(todo.uid, todo.student_answer), total=len(todo)):
    matches = tool.check(str(text))
    buf.append({'uid': uid, 'n_errors': len(matches), 'n_typos': sum(is_typo(m) for m in matches)})
    if len(buf) == 500:
        pd.DataFrame(buf).to_csv(gpath, mode='a', header=not os.path.exists(gpath), index=False); buf = []
if buf:
    pd.DataFrame(buf).to_csv(gpath, mode='a', header=not os.path.exists(gpath), index=False)
gram = pd.read_csv(gpath).drop_duplicates('uid')
print(len(gram), 'answers grammar-checked')

  0%|          | 0/13077 [00:00<?, ?it/s]

13077 answers grammar-checked


In [8]:
from sentence_transformers import SentenceTransformer
import sentence_transformers
versions['sentence_transformers'] = sentence_transformers.__version__
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')

def cosine_pairs(refs, studs):
    e1 = st_model.encode(list(map(str, refs)), batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    e2 = st_model.encode(list(map(str, studs)), batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    return (e1 * e2).sum(1)

pairs = pd.concat([mo[['uid', 'reference_answer', 'student_answer']],
                   se[['uid', 'reference_answer', 'student_answer']]], ignore_index=True)
fpath = f'{ROOT}/cache/features.csv'
if os.path.exists(fpath):
    feats = pd.read_csv(fpath)
else:
    feats = pairs[['uid']].copy()
    feats['cos'] = cosine_pairs(pairs.reference_answer, pairs.student_answer)
    feats['keyword'] = [keyword_score(r, s) for r, s in tqdm(zip(pairs.reference_answer, pairs.student_answer), total=len(pairs))]
    feats['synonym'] = [synonym_score(r, s) for r, s in tqdm(zip(pairs.reference_answer, pairs.student_answer), total=len(pairs))]
    feats = feats.merge(gram, on='uid', how='left')
    assert feats.n_errors.notna().all()
    feats['grammar'] = feats.n_errors.apply(grammar_score)
    feats['context'] = context_score(feats.cos)
    feats['composite_fixed'] = composite_fixed(feats)
    feats.to_csv(fpath, index=False)
feats.describe().round(2)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/205 [00:00<?, ?it/s]

Batches:   0%|          | 0/205 [00:00<?, ?it/s]

  0%|          | 0/13077 [00:00<?, ?it/s]

  0%|          | 0/13077 [00:00<?, ?it/s]

,cos,keyword,synonym,n_errors,n_typos,grammar,context,composite_fixed
count,13077.00,13077.00,13077.00,13077.00,13077.00,13077.00,13077.00,13077.00
mean,0.54,2.66,0.30,0.27,0.05,9.55,5.37,4.21
std,0.21,2.51,0.71,0.69,0.31,1.06,2.12,1.58
min,-0.11,0.00,0.00,0.00,0.00,0.00,0.00,0.53
25%,0.39,0.71,0.00,0.00,0.00,10.00,3.90,3.06
50%,0.55,2.00,0.00,0.00,0.00,10.00,5.50,4.16
75%,0.69,4.00,0.00,0.00,0.00,10.00,6.87,5.20
max,1.00,10.00,10.00,14.00,11.00,10.00,10.00,8.50


## 5. Latency of each lexical component (for the computational-cost discussion)

In [9]:
sample = pairs.sample(200, random_state=SEED)
lat = {}
t = time.perf_counter(); [keyword_score(r, s) for r, s in zip(sample.reference_answer, sample.student_answer)]; lat['keyword'] = time.perf_counter() - t
t = time.perf_counter(); [synonym_score(r, s) for r, s in zip(sample.reference_answer, sample.student_answer)]; lat['synonym'] = time.perf_counter() - t
t = time.perf_counter(); [tool.check(str(s)) for s in sample.student_answer]; lat['grammar'] = time.perf_counter() - t
t = time.perf_counter()
for r, s in zip(sample.reference_answer, sample.student_answer):
    st_model.encode([str(r), str(s)], normalize_embeddings=True)
lat['context_minilm'] = time.perf_counter() - t
latency = pd.DataFrame({'component': list(lat), 'ms_per_answer_cpu': [1000 * v / len(sample) for v in lat.values()]})
latency['minilm_params_M'] = sum(p.numel() for p in st_model.parameters()) / 1e6
latency.to_csv(f'{ROOT}/results/nb0_latency.csv', index=False)
latency.round(2)

,component,ms_per_answer_cpu,minilm_params_M
0,keyword,0.21,22.71
1,synonym,0.25,22.71
2,grammar,41.56,22.71
3,context_minilm,37.57,22.71


## 6. Mohler — out-of-fold predictions for all non-LLM systems

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.feature_extraction.text import TfidfVectorizer

COMP = ['context', 'keyword', 'synonym', 'grammar']
M = mo.merge(feats, on='uid', how='left')
preds = M[['uid', 'group', 'fold', 'y']].copy()
weights = []

def tfidf_cos(fit_df, apply_df):
    vec = TfidfVectorizer(stop_words='english').fit(pd.concat([fit_df.reference_answer, fit_df.student_answer]).astype(str))
    a = vec.transform(apply_df.reference_answer.astype(str)); b = vec.transform(apply_df.student_answer.astype(str))
    num = np.asarray(a.multiply(b).sum(1)).ravel()
    den = np.sqrt(np.asarray(a.multiply(a).sum(1)).ravel() * np.asarray(b.multiply(b).sum(1)).ravel())
    return np.divide(num, den, out=np.zeros_like(num), where=den > 0)

def fit_predict(X_tr, y_tr, X_te, positive=False):
    lr = LinearRegression(positive=positive).fit(X_tr, y_tr)
    return np.clip(lr.predict(X_te), 0, 10), lr

for k in range(N_FOLDS):
    fit = M[M[f'split_f{k}'] != 'test']        # train + val (no hyperparameters to tune here)
    te = M[M[f'split_f{k}'] == 'test']
    idx = te.index
    preds.loc[idx, 'mean'] = fit.y.mean()
    tf_fit, tf_te = tfidf_cos(fit, fit), tfidf_cos(fit, te)
    preds.loc[idx, 'tfidf_cos'] = fit_predict(tf_fit[:, None], fit.y, tf_te[:, None])[0]
    preds.loc[idx, 'minilm_cos'] = fit_predict(fit[['context']], fit.y, te[['context']])[0]
    preds.loc[idx, 'composite_fixed'] = te['composite_fixed'].clip(0, 10)            # original Eq. 6, uncalibrated
    preds.loc[idx, 'composite_fixed_cal'] = fit_predict(fit[['composite_fixed']], fit.y, te[['composite_fixed']])[0]
    p, lr = fit_predict(fit[COMP], fit.y, te[COMP], positive=True)
    preds.loc[idx, 'composite_learned'] = p
    w = lr.coef_ / lr.coef_.sum() if lr.coef_.sum() > 0 else lr.coef_
    weights.append({'fold': k, **dict(zip(COMP, w)), 'intercept': lr.intercept_})
    for c in COMP:                               # leave-one-component-out ablation
        keep = [x for x in COMP if x != c]
        preds.loc[idx, f'composite_learned_minus_{c}'] = fit_predict(fit[keep], fit.y, te[keep], positive=True)[0]

weights = pd.DataFrame(weights)
weights.to_csv(f'{ROOT}/results/nb0_weights.csv', index=False)
preds.to_csv(f'{ROOT}/results/mohler_preds.csv', index=False)
print('Learned (normalised) composite weights per fold:'); display(weights.round(3))

Learned (normalised) composite weights per fold:


,fold,context,keyword,synonym,grammar,intercept
0,0,0.415,0.284,0.260,0.041,5.570
1,1,0.443,0.225,0.264,0.067,5.038
2,2,0.437,0.263,0.238,0.063,5.357
3,3,0.415,0.306,0.232,0.046,5.626
4,4,0.430,0.326,0.221,0.023,5.498


In [11]:
systems = [c for c in preds.columns if c not in ['uid', 'group', 'fold', 'y']]
rows = []
for s in tqdm(systems):
    rows.append({'system': s, **summarise(preds.y, preds[s], reg_metrics, groups=preds.group, folds=preds.fold)})
human_row = {'system': 'human_grader1_vs_grader2', **{k: v for k, v in human.items()}}
mo_summary = pd.DataFrame(rows + [human_row])
# a per-fold constant has no meaningful correlation, so only MAE/RMSE are reported for it
corr_cols = [c for c in mo_summary.columns if c.startswith(('pearson', 'spearman', 'qwk'))]
mo_summary.loc[mo_summary.system == 'mean', corr_cols] = np.nan
mo_summary.to_csv(f'{ROOT}/results/mohler_nb0_summary.csv', index=False)
mo_summary[['system', 'pearson', 'pearson_ci', 'spearman', 'mae', 'rmse', 'qwk', 'pearson_fold_std']].round(3)

  0%|          | 0/10 [00:00<?, ?it/s]

,system,pearson,pearson_ci,spearman,mae,rmse,qwk,pearson_fold_std
0,mean,NaN,NaN,NaN,1.803,2.215,NaN,NaN
1,tfidf_cos,0.407,"[0.341, 0.473]",0.434,1.543,2.023,0.304,0.057
2,minilm_cos,0.486,"[0.425, 0.545]",0.478,1.474,1.935,0.375,0.093
3,composite_fixed,0.542,"[0.491, 0.590]",0.571,3.856,4.253,0.185,0.079
4,composite_fixed_cal,0.548,"[0.491, 0.601]",0.565,1.388,1.856,0.433,0.086
5,composite_learned,0.543,"[0.484, 0.597]",0.558,1.394,1.862,0.428,0.089
6,composite_learned_minus_context,0.491,"[0.426, 0.552]",0.538,1.442,1.933,0.368,0.087
7,composite_learned_minus_keyword,0.486,"[0.424, 0.544]",0.479,1.474,1.935,0.377,0.092
8,composite_learned_minus_synonym,0.538,"[0.481, 0.593]",0.553,1.404,1.869,0.425,0.092
9,composite_learned_minus_grammar,0.543,"[0.485, 0.596]",0.557,1.393,1.862,0.430,0.088


## 7. SciEntsBank — classifiers on the same features, per labelling scheme

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

S = se.merge(feats, on='uid', how='left')
fit = S[S.split.isin(['train', 'val'])]
TESTS = ['test_ua', 'test_uq', 'test_ud']
S['tfidf_cos'] = tfidf_cos(fit, S)
fit = S[S.split.isin(['train', 'val'])]

feature_sets = {'tfidf_cos': ['tfidf_cos'], 'minilm_cos': ['context'],
                'composite_fixed': ['composite_fixed'], 'lexical_4': COMP}
for c in COMP:
    feature_sets[f'lexical_4_minus_{c}'] = [x for x in COMP if x != c]

label_col = {'5way': 'label', '3way': 'label3', '2way': 'label2'}
se_preds = S[['uid', 'group', 'split', 'label', 'label3', 'label2']].copy()
for scheme, col in label_col.items():
    se_preds[f'majority_{scheme}'] = fit[col].mode()[0]
    for name, cols in feature_sets.items():
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(fit[cols], fit[col])
        se_preds[f'{name}_{scheme}'] = clf.predict(S[cols])
se_preds.to_csv(f'{ROOT}/results/scientsbank_preds.csv', index=False)

rows = []
for scheme, col in label_col.items():
    for name in ['majority'] + list(feature_sets):
        for t in TESTS:
            d = se_preds[se_preds.split == t]
            groups = None if t == 'test_ua' else d.group      # UQ/UD: resample whole questions
            rows.append({'scheme': scheme, 'system': name, 'test': t,
                         **summarise(d[col], d[f'{name}_{scheme}'], cls_metrics, groups=groups)})
se_summary = pd.DataFrame(rows)
se_summary.to_csv(f'{ROOT}/results/scientsbank_nb0_summary.csv', index=False)
se_summary.pivot_table(index=['scheme', 'system'], columns='test', values='macro_f1').round(3)

test                            test_ua  test_ud  test_uq
scheme system                                            
2way   composite_fixed            0.659    0.681    0.700
       lexical_4                  0.666    0.690    0.685
       lexical_4_minus_context    0.662    0.688    0.679
       lexical_4_minus_grammar    0.656    0.689    0.691
       lexical_4_minus_keyword    0.632    0.649    0.665
       lexical_4_minus_synonym    0.648    0.684    0.708
       majority                   0.362    0.367    0.371
       minilm_cos                 0.619    0.648    0.666
       tfidf_cos                  0.609    0.653    0.670
3way   composite_fixed            0.419    0.439    0.435
       lexical_4                  0.417    0.442    0.450
       lexical_4_minus_context    0.419    0.441    0.423
       lexical_4_minus_grammar    0.414    0.442    0.446
       lexical_4_minus_keyword    0.417    0.424    0.416
       lexical_4_minus_synonym    0.417    0.438    0.450
       majority                   0.210    0.219    0.223
       minilm_cos                 0.410    0.424    0.417
       tfidf_cos                  0.401    0.416    0.431
5way   composite_fixed            0.228    0.224    0.289
       lexical_4                  0.253    0.265    0.294
       lexical_4_minus_context    0.232    0.229    0.283
       lexical_4_minus_grammar    0.249    0.265    0.294
       lexical_4_minus_keyword    0.229    0.221    0.279
       lexical_4_minus_synonym    0.250    0.261    0.291
       majority                   0.121    0.118    0.146
       minilm_cos                 0.220    0.217    0.280
       tfidf_cos                  0.216    0.223    0.264

In [13]:
json.dump(versions, open(f'{ROOT}/results/env_nb0.json', 'w'), indent=2)
print(versions)

{'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.2.3', 'sklearn': '1.6.1', 'nltk': '3.9.1', 'language_tool_python': '3.4.0', 'sentence_transformers': '5.7.0'}
